# Task 5 — controlled GMM ranking probe at the v9 alert count

The scored v9 submission has **0.74593 overall** with **6,531** Task 5 alerts from the engineered-feature OCSVM. The previous 35% OCSVM draft remains unscored. This candidate changes the detector ranking to the saved engineered-feature Gaussian mixture while keeping **exactly 6,531** alerts, the confirmed `1 = anomaly` orientation, and every Task 1–4 value equal to v9. Thus Kaggle feedback can isolate the ranking swap from alert volume.

`rank_diagnostics_v1.ipynb` found 1,151 exclusive accounts per detector at a fixed top-25% count; GMM-only accounts have more supplied security flags. Those flags are model inputs and not labels. This notebook creates a local submission draft only. The lecturer reportedly permits unsupervised Task 5 fitting on the provided Test-only file although the written rule differs.

In [1]:
from pathlib import Path
from datetime import datetime,timezone
from hashlib import sha256
import copy,json,sys,joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/'src'/'pipeline'/'save_data.py').exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
SRC=ROOT/'src'
import numpy as np,pandas as pd
from sklearn.base import clone
from sklearn.model_selection import train_test_split
from src.pipeline.random_seed import set_seed
from src.pipeline.save_model import save_model
from src.pipeline.evaluate_model import evaluate_model
from src.pipeline.build_sample_submission import build_sample_submission
SEED=set_seed()
DATA_NAME='eda_1_v2'
BASE_MODEL='t5_r2_gmm_engineered_v2'
MODEL_NAME='t5_r6_gmm_match_v9_alerts_v6'
BASE_SUBMISSION='kaggle_t12345_t5_alert25_v9'
SUBMISSION_NAME='kaggle_t12345_t5_gmm_match_v9_v11'
card=json.loads((SRC/'task5'/'data'/'eda'/f'{DATA_NAME}.json').read_text(encoding='utf8'))
assert card['name']==DATA_NAME and card['task']=='task5' and set(card['splits'])=={'test'}
test_path=SRC/card['splits']['test']['path']
test=pd.read_parquet(test_path) if card['format']=='parquet' else pd.read_csv(test_path)
assert test.player_id.is_unique and 'is_anomaly' not in test
X=test.drop(columns='player_id')
model_card=json.loads((SRC/'task5'/'save'/f'{BASE_MODEL}.json').read_text(encoding='utf8'))
assert model_card['data_name']==DATA_NAME
base_model=joblib.load(SRC/model_card['model_path'])
base_card=json.loads((SRC/'submission'/f'{BASE_SUBMISSION}.json').read_text(encoding='utf8'))
base_path=SRC/base_card['submission_path']
assert sha256(base_path.read_bytes()).hexdigest()==base_card['sha256']
base=pd.read_csv(base_path)
sample=pd.read_csv(SRC/'submission'/'sample_submission.csv')
assert base.columns.tolist()==sample.columns.tolist()==['id','task1','task2','task3','task4','task5']
assert base.id.tolist()==sample.id.tolist() and len(base)==len(sample)==len(test)==25889
assert base.task5.sum()==6531
K=int(base.task5.sum())
print('Matched v9 alert count:',K)

Matched v9 alert count: 6531


## Save one detector-ranking candidate

The GMM was already fitted on the original 12,000-row unsupervised split with fitted imputer and scaler. Reuse it unchanged, selecting the cutoff between the 6,531st and 6,532nd highest scores on the provided unlabeled file. This guarantees the same alert count as v9 and does not use labels. An overlapping 10,000-row GMM fit supplies a top-25% rank-stability proxy on the previously defined 4,000-row diagnostic holdout; it is not F3.

In [2]:
scores=np.asarray(base_model.anomaly_score(X),dtype=float)
assert scores.shape==(len(X),) and np.isfinite(scores).all()
ordered=np.sort(scores)
assert ordered[-K]>ordered[-K-1], 'Inspect score tie at exact alert budget'
threshold=float((ordered[-K]+ordered[-K-1])/2)
SAVE=SRC/'task5'/'save'
model_path,model_card_path,eval_path=(SAVE/f'{MODEL_NAME}.joblib',SAVE/f'{MODEL_NAME}.json',SAVE/f'{MODEL_NAME}-evaluation.json')
if not (model_path.exists() and model_card_path.exists() and eval_path.exists()):
    assert not any(p.exists() for p in (model_path,model_card_path,eval_path)),'Inspect incomplete artifact'
    candidate=copy.deepcopy(base_model)
    candidate.threshold_=threshold
    candidate.anomaly_fraction=K/len(X)
    assert np.array_equal(candidate.anomaly_score(X),scores)
    rows=np.arange(len(X))
    dev_idx,hold_idx=train_test_split(rows,train_size=12000,test_size=4000,random_state=SEED)
    dev_b_idx=np.random.default_rng(SEED).choice(dev_idx,size=10000,replace=False)
    alternate=clone(base_model).fit(X.iloc[dev_b_idx])
    a=base_model.anomaly_score(X.iloc[hold_idx]);b=alternate.anomaly_score(X.iloc[hold_idx])
    k=round(.25*len(hold_idx))
    top_a=set(np.argpartition(a,-k)[-k:]);top_b=set(np.argpartition(b,-k)[-k:])
    stability=len(top_a&top_b)/len(top_a|top_b)
    save_model(task=5,name=MODEL_NAME,model=candidate,data_name=DATA_NAME,
        card={'idea':'Saved GMM ranking with exact v9 alert count',
              'base_model':BASE_MODEL,'matched_submission':BASE_SUBMISSION,
              'alert_count':K,'threshold_source':'unlabeled full-file score rank',
              'seed':SEED,'scope':'user-reported lecturer permission for unsupervised Task 5 Test fit'})
    result=evaluate_model(task=5,name=MODEL_NAME,score=stability,
        metric_name='top25_rank_stability_jaccard',higher_is_better=True,
        card={'meaning':'Top-25% rank overlap between overlapping GMM fits on unlabeled rows; not F3',
              'seed':SEED})
    print('Top-25% stability proxy:',result['score'])
else:
    saved_card=json.loads(model_card_path.read_text(encoding='utf8'))
    assert saved_card['data_name']==DATA_NAME and saved_card['card']['matched_submission']==BASE_SUBMISSION
    candidate=joblib.load(model_path)
    assert candidate.threshold_==threshold
    print('Loaded saved candidate and proxy:',json.loads(eval_path.read_text(encoding='utf8'))['score'])
pred=np.asarray(candidate.predict(X),dtype=int)
assert pred.shape==(len(X),) and set(pred)=={0,1} and pred.sum()==K
prior=base.task5.to_numpy(dtype=int)
added=int(((pred==1)&(prior==0)).sum())
removed=int(((pred==0)&(prior==1)).sum())
assert added==removed and added>0
print('Swapped alerts versus v9:',{'added':added,'removed':removed,'unchanged_positive':K-removed})
flags=X[['password_changed','vpn_usage','suspicious_login_time','mass_item_sale']].fillna(0).gt(0).any(axis=1).to_numpy()
for label,mask in [('GMM-only',(pred==1)&(prior==0)),('OCSVM-only',(pred==0)&(prior==1))]:
    print(label,'rows',int(mask.sum()),'security flag rate',float(flags[mask].mean()))

Top-25% stability proxy: 0.9841269841269841
Swapped alerts versus v9: {'added': 1154, 'removed': 1154, 'unchanged_positive': 5377}
GMM-only rows 1154 security flag rate 0.7027729636048526
OCSVM-only rows 1154 security flag rate 0.12218370883882149


## Build the controlled submission draft

Create the per-task artifact through the pipeline, then copy Tasks 1–4 and ANS IDs from the scored v9 CSV. Verify the exact alert count and record both swapped groups in the final card. No Kaggle upload occurs.

In [3]:
part_path=SRC/'task5'/'submission'/f'{MODEL_NAME}.csv'
part_card_path=SRC/'task5'/'submission'/f'{MODEL_NAME}.json'
if not (part_path.exists() and part_card_path.exists()):
    assert not (part_path.exists() or part_card_path.exists()),'Inspect incomplete per-task submission'
    build_sample_submission(task=5,name=MODEL_NAME,model_name=MODEL_NAME,id_column='player_id',
        card={'idea':'GMM ranking at v9 alert count','scope':'Test row order mapped to ANS IDs in final assembly'})
part=pd.read_csv(part_path,sep='|')
assert part.columns.tolist()==['id','task5'] and part.id.tolist()==test.player_id.tolist()
assert np.array_equal(part.task5.to_numpy(dtype=int),pred)
output=base.copy();output['task5']=pred
assert output[['id','task1','task2','task3','task4']].equals(base[['id','task1','task2','task3','task4']])
assert output.id.tolist()==sample.id.tolist() and output.notna().all().all()
assert int(output.task5.sum())==K
out_path=SRC/'submission'/f'{SUBMISSION_NAME}.csv'
out_card_path=SRC/'submission'/f'{SUBMISSION_NAME}.json'
if not (out_path.exists() and out_card_path.exists()):
    assert not (out_path.exists() or out_card_path.exists()),'Inspect incomplete final submission'
    output.to_csv(out_path,index=False)
    sources=dict(base_card['sources'])
    sources['task5']={'model_name':MODEL_NAME,'model_card_path':f'task5/save/{MODEL_NAME}.json',
        'model_path':f'task5/save/{MODEL_NAME}.joblib','data_name':DATA_NAME,
        'data_card_path':f'task5/data/eda/{DATA_NAME}.json',
        'test_path':card['splits']['test']['path'],
        'evaluation_metric':'top25_rank_stability_jaccard',
        'evaluation_score':json.loads(eval_path.read_text(encoding='utf8'))['score']}
    detail={'name':SUBMISSION_NAME,'created_at':datetime.now(timezone.utc).isoformat(),
        'card':'Task 5 GMM versus OCSVM ranking at identical alert count; Tasks 1-4 fixed',
        'submission_path':f'submission/{SUBMISSION_NAME}.csv',
        'format':'comma-separated CSV','rows':len(output),'columns':output.columns.tolist(),
        'sha256':sha256(out_path.read_bytes()).hexdigest(),
        'base_submission':BASE_SUBMISSION,'base_sha256':base_card['sha256'],
        'changed_tasks':[5],'task5_alerts':K,'task5_added_alerts':added,
        'task5_removed_alerts':removed,
        'task5_metric_note':'Unlabeled rank stability is not macro F3',
        'task5_id_mapping':'Saved EDA Test row position to sample ANS ID',
        'sources':sources}
    out_card_path.write_text(json.dumps(detail,ensure_ascii=False,indent=2)+'\n',encoding='utf8')
saved=pd.read_csv(out_path);saved_card=json.loads(out_card_path.read_text(encoding='utf8'))
assert saved.equals(output) and saved_card['sha256']==sha256(out_path.read_bytes()).hexdigest()
assert saved_card['base_sha256']==base_card['sha256']
print('Prepared CSV:',out_path)
print('SHA-256:',saved_card['sha256'])
print('Exactly matched alerts:',K,'; swapped',added,'in and',removed,'out')

Prepared CSV: C:\Users\BTCOM\Desktop\ml-final\src\submission\kaggle_t12345_t5_gmm_match_v9_v11.csv
SHA-256: 588aceae805088367ddf1766fc76d927603a4ff61b4472a236f91eb7442abecd
Exactly matched alerts: 6531 ; swapped 1154 in and 1154 out
